# OOF Stacking + PatchTST 기반 수요예측 보정 모델

## 방법론 개요

### 1. OOF (Out-of-Fold) Stacking
K-Fold 교차검증을 통해 **Out-of-Sample 예측**을 수집하고, 이를 메타 피처로 Ridge 회귀를 학습하는 2단계 앙상블 방법론.

| 구분 | Residual 방식 | OOF 방식 |
|------|-------------|----------|
| 메타 입력 | In-sample 잔차 | Out-of-sample 예측값 |
| 과적합 위험 | 높음 | 낮음 (홀드아웃 예측 사용) |
| 메타 학습 데이터 | 전체 학습셋 | OOF 유효 구간 |

**OOF 구현 절차**
```
for fold k in TimeSeriesSplit(K=5):
    fold_train → Base Models 학습 (fold-specific scaler 적용)
    fold_val   → OOF 예측 수집  (Out-of-sample)

Meta Model: Ridge( [OOF_ND, OOF_GB, OOF_LGB, OOF_PatchTST] ) → y_train

Inference:  전체 학습데이터로 Base Models 재학습 → Meta Model 적용
```

### 2. PatchTST (Nie et al., 2023)
"A Time Series is Worth 64 Words: Long-term Forecasting with Transformers"

- **Patch 토큰화**: 시계열을 `patch_len` 크기 패치로 분할 → 지역적 의미 단위 보존, 시퀀스 길이 축소
- **Channel-Independence**: 각 변수를 독립적으로 동일한 Transformer 처리 → 파라미터 효율
- **Pre-Layer Normalization**: Pre-LN Transformer로 학습 안정성 향상

```
Input (B, L, C)  →  Patch 분할 (B×C, n_p, p)  →  Transformer Encoder
                 →  채널 평균 집계  →  Prediction Head  →  (B,)
```
하이퍼파라미터: `seq_len=26, patch_len=4, stride=2` → `n_patches=12`

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble        import GradientBoostingRegressor
from sklearn.linear_model    import Ridge
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import mean_squared_error, mean_absolute_error

import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

In [ ]:
# ── 공통 설정 ─────────────────────────────────────────────────────────────
DATA_PATH     = 'data_weekly_260120.csv'
TARGET_COL    = 'Com_LME_Ni_Cash'

VAL_START     = pd.to_datetime('2025-08-04')
VAL_END       = pd.to_datetime('2025-10-20')
TEST_START    = pd.to_datetime('2025-10-27')
TEST_END      = pd.to_datetime('2026-01-12')

N_SPLITS      = 5       # OOF K-Fold 수
RANDOM_STATE  = 42
BASELINE_RMSE = 406.80  # sparta2_advanced.ipynb 기준

# ── PatchTST 설정 ─────────────────────────────────────────────────────────
SEQ_LEN      = 26   # 입력 시퀀스 길이 (26주 ≈ 6개월)
PATCH_LEN    = 4    # 패치 크기 (4주 ≈ 1개월)
STRIDE       = 2    # 슬라이딩 스텝  →  n_patches = (26-4)//2 + 1 = 12
D_MODEL      = 32
N_HEAD       = 4
N_LAYERS     = 2
DROPOUT      = 0.1
LR           = 5e-4
BATCH_SIZE   = 32
OOF_EPOCHS   = 40   # OOF 폴드: 고정 에포크
FINAL_EPOCHS = 100  # 최종 모델: Early Stopping 적용
PATIENCE     = 15

## 1. 데이터 로딩 · 피처 엔지니어링 · 분할

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['dt']).set_index('dt').sort_index()
print(f'기간: {df.index[0].date()} ~ {df.index[-1].date()}  |  shape: {df.shape}')

# ── 피처 엔지니어링 (sparta2_advanced.ipynb 동일 기준) ────────────────────
feat_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feat_cols].shift(1)   # 1주 lag: leakage 방지
y = df[TARGET_COL]

log_ret = np.log(y / y.shift(1))
for w in [4, 8, 12, 26]:
    X[f'RV_{w}w']      = log_ret.shift(1).rolling(w).std() * np.sqrt(52)
for w in [4, 12, 26]:
    X[f'ROC_{w}w']     = y.shift(1).pct_change(w)
for w in [4, 26]:
    mu, sig            = y.shift(1).rolling(w).mean(), y.shift(1).rolling(w).std()
    X[f'zscore_{w}w']  = (y.shift(1) - mu) / (sig + 1e-8)
for lag in [1, 2, 3]:
    X[f'ret_lag_{lag}'] = log_ret.shift(lag)

valid = X.notna().all(axis=1) & y.notna()
X, y  = X[valid], y[valid]
N_VARS = X.shape[1]
print(f'유효 샘플: {len(X)}  |  피처: {N_VARS}')

# ── 분할 ─────────────────────────────────────────────────────────────────
tr = X.index < VAL_START
va = (X.index >= VAL_START)  & (X.index <= VAL_END)
te = (X.index >= TEST_START) & (X.index <= TEST_END)

X_train, y_train = X[tr], y[tr]
X_val,   y_val   = X[va], y[va]
X_test,  y_test  = X[te], y[te]

for tag, d in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {tag:5s}: {len(d):4d}개  ({d.index[0].date()} ~ {d.index[-1].date()})')

## 2. 공통 유틸리티

In [ ]:
def eval_metrics(y_true, y_pred, name=''):
    """RMSE / MAE / MAPE 출력 후 dict 반환"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    print(f'  [{name:42s}]  RMSE={rmse:7.2f}  MAE={mae:7.2f}  MAPE={mape:.2f}%')
    return dict(model=name, rmse=rmse, mae=mae, mape=mape)


def naive_drift(y_series, n_steps):
    """1차 Drift 다단계 예측: y_{t+k} = y_t + k*(y_t - y_{t-1})"""
    last, drift = y_series.iloc[-1], y_series.iloc[-1] - y_series.iloc[-2]
    return np.array([last + (k + 1) * drift for k in range(n_steps)])


def make_sequences(X_arr, y_arr, seq_len):
    """
    슬라이딩 윈도우 시퀀스 생성.
    X_seqs[k] = X_arr[k : k+seq_len]  (seq_len, n_vars)
    y_seqs[k] = y_arr[k + seq_len]     스칼라 타겟
    """
    n = len(X_arr)
    X_seqs = np.stack([X_arr[i:i + seq_len] for i in range(n - seq_len)])
    y_seqs = y_arr[seq_len:]
    return X_seqs, y_seqs

## 3. PatchTST 모델 정의

**Nie et al. (2023)** 의 Channel-Independence PatchTST를 단일 타겟 예측에 맞게 구현.

| 하이퍼파라미터 | 값 | 의미 |
|---|---|---|
| seq_len | 26 | 입력 길이 (주) |
| patch_len | 4 | 패치 크기 (≈1개월) |
| stride | 2 | 슬라이딩 스텝 |
| n_patches | 12 | `(26-4)//2 + 1` |
| d_model | 32 | 임베딩 차원 |
| nhead | 4 | 어텐션 헤드 수 |
| n_layers | 2 | Encoder 층 수 |

In [ ]:
class PatchTST(nn.Module):
    """
    PatchTST: A Time Series is Worth 64 Words (Nie et al., 2023)

    Channel-Independence 방식 구현:
      - 각 입력 변수(채널)를 독립적으로 동일한 Transformer에 통과
      - 채널 평균 집계 후 단일 타겟 예측
      - Pre-LN TransformerEncoderLayer 로 학습 안정성 확보
    """
    def __init__(self, n_vars, seq_len,
                 patch_len=4, stride=2,
                 d_model=32, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        n_patches       = (seq_len - patch_len) // stride + 1
        self.patch_len  = patch_len
        self.stride     = stride

        # Patch linear projection (공유 파라미터)
        self.patch_proj = nn.Linear(patch_len, d_model)

        # 학습 가능한 위치 인코딩
        self.pos_enc    = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            norm_first=True               # Pre-LN
        )
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.dropout    = nn.Dropout(dropout)
        self.head       = nn.Linear(d_model * n_patches, 1)

    def forward(self, x):
        # x : (B, seq_len, n_vars)
        B, L, C = x.shape

        # Patch 분할: permute → unfold → (B, C, n_patches, patch_len)
        x   = x.permute(0, 2, 1).unfold(2, self.patch_len, self.stride)
        n_p = x.shape[2]

        # 채널 독립: (B*C, n_patches, patch_len)
        x = x.reshape(B * C, n_p, self.patch_len)

        # Patch 임베딩 + 위치 인코딩
        x = self.dropout(self.patch_proj(x) + self.pos_enc)  # (B*C, n_p, d_model)

        # Transformer Encoder
        x = self.encoder(x)                    # (B*C, n_p, d_model)

        # 채널 평균 집계 후 예측
        x = x.reshape(B, C, -1).mean(dim=1)    # (B, n_p * d_model)
        return self.head(x).squeeze(-1)         # (B,)


def _train_patchtst(model, X_tr, y_tr, epochs,
                    X_val=None, y_val=None, patience=None):
    """
    PatchTST 학습.
    X_val/y_val + patience 제공 시 Early Stopping 적용, 미제공 시 고정 에포크.
    """
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    ds  = TensorDataset(torch.FloatTensor(X_tr).to(DEVICE),
                        torch.FloatTensor(y_tr).to(DEVICE))
    dl  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

    use_es   = (X_val is not None) and (patience is not None)
    best_loss, best_state, cnt = float('inf'), None, 0

    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            nn.MSELoss()(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        if use_es:
            model.eval()
            with torch.no_grad():
                Xv  = torch.FloatTensor(X_val).to(DEVICE)
                yv  = torch.FloatTensor(y_val).to(DEVICE)
                val_loss = nn.MSELoss()(model(Xv), yv).item()
            if val_loss < best_loss:
                best_loss  = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                cnt        = 0
            else:
                cnt += 1
                if cnt >= patience:
                    break

    if use_es and best_state:
        model.load_state_dict(best_state)
    return model


N_PATCHES = (SEQ_LEN - PATCH_LEN) // STRIDE + 1
print(f'PatchTST  seq_len={SEQ_LEN}  patch_len={PATCH_LEN}  '
      f'stride={STRIDE}  n_patches={N_PATCHES}  d_model={D_MODEL}')

## 4. 베이스라인 모델 (비교 기준)

In [ ]:
results = []   # 최종 성능 누적

# Naive Drift (결정적; 전체 학습셋 마지막 두 값 사용)
nd_val  = naive_drift(y_train, len(y_val))
nd_test = naive_drift(y_val,   len(y_test))

# GradientBoosting
gb_final = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE
).fit(X_train.values, y_train.values)

# LightGBM
lgb_final = lgb.LGBMRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    random_state=RANDOM_STATE, verbose=-1
).fit(X_train.values, y_train.values)

gb_test  = gb_final.predict(X_test.values)
lgb_test = lgb_final.predict(X_test.values)

print('=== 베이스라인 성능 (Test Set) ===')
results.append(eval_metrics(y_test, nd_test,                        'Naive Drift'))
results.append(eval_metrics(y_test, gb_test,                        'GradientBoosting'))
results.append(eval_metrics(y_test, lgb_test,                       'LightGBM'))
results.append(eval_metrics(y_test, 0.8*nd_test + 0.2*gb_test,     'Hybrid (0.8 Naive + 0.2 GB)'))
print(f'\n  sparta2_advanced 기준선 RMSE = {BASELINE_RMSE}')

## 5. OOF 예측 생성

**OOF 정확성 보장 원칙**
1. `TimeSeriesSplit` — 과거→미래 방향 고정, 시간적 leakage 원천 차단
2. **Fold별 독립 스케일러** — `StandardScaler`를 `fold_train`에만 fit, `fold_val`에 transform
3. **시퀀스 경계** — PatchTST 시퀀스 생성 시 fold_val의 타겟에 해당하는 슬라이스만 사용
4. **Naive Drift** — 각 폴드 학습셋 마지막 두 값으로 drift 산출, 인퍼런스와 동일한 공식 적용

In [ ]:
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)
X_tr_arr = X_train.values
y_tr_arr = y_train.values
N_TR     = len(X_train)

oof_nd   = np.zeros(N_TR)
oof_gb   = np.zeros(N_TR)
oof_lgb  = np.zeros(N_TR)
oof_ptst = np.zeros(N_TR)
oof_mask = np.zeros(N_TR, dtype=bool)   # OOF 예측이 채워진 인덱스

print(f'OOF 생성 시작  (K={N_SPLITS}, SEQ_LEN={SEQ_LEN})\n')

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_tr_arr)):
    n_tr, n_val = len(tr_idx), len(val_idx)
    print(f'  Fold {fold+1}/{N_SPLITS}  '
          f'train={n_tr}  val={n_val}  '
          f'[{X_train.index[val_idx[0]].date()} ~ {X_train.index[val_idx[-1]].date()}]')

    X_fold_tr  = X_tr_arr[tr_idx]
    X_fold_val = X_tr_arr[val_idx]
    y_fold_tr  = y_tr_arr[tr_idx]
    y_fold_val = y_tr_arr[val_idx]

    # ── Naive Drift OOF ─────────────────────────────────────────────────
    # 인퍼런스와 동일한 공식: y_{t+k} = y_t + k*(y_t - y_{t-1})
    last, drift         = y_fold_tr[-1], y_fold_tr[-1] - y_fold_tr[-2]
    oof_nd[val_idx]     = [last + (k + 1) * drift for k in range(n_val)]

    # ── GradientBoosting OOF (스케일링 불필요) ───────────────────────────
    gb_fold = GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE
    ).fit(X_fold_tr, y_fold_tr)
    oof_gb[val_idx]     = gb_fold.predict(X_fold_val)

    # ── LightGBM OOF ────────────────────────────────────────────────────
    lgb_fold = lgb.LGBMRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=RANDOM_STATE, verbose=-1
    ).fit(X_fold_tr, y_fold_tr)
    oof_lgb[val_idx]    = lgb_fold.predict(X_fold_val)

    # ── PatchTST OOF (fold-specific scaler) ─────────────────────────────
    if n_tr > SEQ_LEN:   # 시퀀스 생성에 충분한 학습 샘플 필요

        # fold_train 에만 fit → fold_val 에 transform (leakage 없음)
        xs = StandardScaler().fit(X_fold_tr)
        ys = StandardScaler().fit(y_fold_tr.reshape(-1, 1))

        X_comb_s = xs.transform(np.vstack([X_fold_tr, X_fold_val]))
        y_comb_s = ys.transform(
            np.concatenate([y_fold_tr, y_fold_val]).reshape(-1, 1)
        ).ravel()

        # 시퀀스 경계:
        #   X_seqs[k].target = y_comb_s[k + SEQ_LEN]
        #   fold_train 타겟: k + SEQ_LEN < n_tr  →  k < n_tr - SEQ_LEN
        #   fold_val   타겟: k >= n_tr - SEQ_LEN  (정확히 n_val 개)
        X_seqs, y_seqs = make_sequences(X_comb_s, y_comb_s, SEQ_LEN)
        n_tr_seqs      = n_tr - SEQ_LEN

        X_dl_tr = X_seqs[:n_tr_seqs];  y_dl_tr = y_seqs[:n_tr_seqs]
        X_dl_val_fold   = X_seqs[n_tr_seqs:]   # n_val 개, fold_val 타겟과 1:1 대응

        model = PatchTST(N_VARS, SEQ_LEN, PATCH_LEN, STRIDE,
                         D_MODEL, N_HEAD, N_LAYERS, DROPOUT)
        _train_patchtst(model, X_dl_tr, y_dl_tr, OOF_EPOCHS)

        model.eval()
        with torch.no_grad():
            pred_s = model(
                torch.FloatTensor(X_dl_val_fold).to(DEVICE)
            ).cpu().numpy()

        oof_ptst[val_idx] = ys.inverse_transform(pred_s.reshape(-1, 1)).ravel()

    else:
        # 학습 샘플 부족: Naive Drift 대체 (메타 모델이 가중치를 낮게 조정)
        oof_ptst[val_idx] = oof_nd[val_idx]

    oof_mask[val_idx] = True

print(f'\nOOF 완료  |  유효 샘플: {oof_mask.sum()} / {N_TR}')

In [ ]:
# ── OOF 내부 성능 확인 ────────────────────────────────────────────────────
y_oof = y_tr_arr[oof_mask]

print('=== OOF 내부 성능 (학습셋 내 K-Fold, Out-of-Sample) ===')
for arr, tag in [(oof_nd,   'Naive Drift'),
                 (oof_gb,   'GradientBoosting'),
                 (oof_lgb,  'LightGBM'),
                 (oof_ptst, 'PatchTST')]:
    eval_metrics(y_oof, arr[oof_mask], f'OOF {tag}')

# 시각화
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, arr, tag in zip(axes,
                        [oof_nd, oof_gb, oof_lgb, oof_ptst],
                        ['Naive Drift', 'GradientBoosting', 'LightGBM', 'PatchTST']):
    ax.plot(y_oof,           color='black', alpha=0.7, linewidth=1, label='실제값')
    ax.plot(arr[oof_mask],   alpha=0.8,                             label='OOF 예측')
    ax.set_title(f'OOF — {tag}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('OOF 인덱스')
    ax.set_ylabel('Nickel Price ($/ton)')
    ax.grid(alpha=0.3)

plt.suptitle('OOF 예측 vs 실제값 (학습셋 내 K-Fold 교차검증)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ridge 메타 모델

4개 Base Model의 OOF 예측을 입력으로, 실제 니켈 가격을 타겟으로 Ridge 회귀 학습.  
Ridge의 L2 정규화는 Base Model 간 다중공선성을 억제하고 안정적인 가중치를 도출한다.

In [ ]:
X_meta = np.column_stack([
    oof_nd[oof_mask], oof_gb[oof_mask],
    oof_lgb[oof_mask], oof_ptst[oof_mask]
])
y_meta = y_tr_arr[oof_mask]

meta = Ridge(alpha=1.0).fit(X_meta, y_meta)

print('=== Ridge 메타 모델 학습 결과 ===')
print(f'  학습 샘플: {len(y_meta)}  |  alpha=1.0')
print()
print('  앙상블 가중치 (학습 결과):')
for name, c in zip(['Naive Drift', 'GradientBoosting', 'LightGBM', 'PatchTST'], meta.coef_):
    print(f'    {name:20s}: {c:+.6f}')
print(f'    {"Intercept":20s}: {meta.intercept_:+.6f}')
print(f'\n  가중치 합: {meta.coef_.sum():.6f}')

## 7. 최종 예측 (Val / Test)

**Base 모델 전체 학습 데이터로 재학습** 후 OOF 메타 모델 적용.  
PatchTST는 train+val+test 통합 배열로 시퀀스를 구성하여 컨텍스트 경계 문제를 방지.

In [ ]:
# ── PatchTST 최종 학습 (전체 train, Early Stopping on Val) ─────────────
xs_all = StandardScaler().fit(X_train.values)
ys_all = StandardScaler().fit(y_train.values.reshape(-1, 1))

X_tr_s   = xs_all.transform(X_train.values)
X_val_s  = xs_all.transform(X_val.values)
X_test_s = xs_all.transform(X_test.values)
y_tr_s   = ys_all.transform(y_train.values.reshape(-1, 1)).ravel()
y_val_s  = ys_all.transform(y_val.values.reshape(-1, 1)).ravel()
y_test_s = ys_all.transform(y_test.values.reshape(-1, 1)).ravel()

# Train + Val + Test 통합 배열로 시퀀스 생성
# → Val/Test 시퀀스가 Train 컨텍스트를 자연스럽게 포함
X_all = np.vstack([X_tr_s, X_val_s, X_test_s])
y_all = np.concatenate([y_tr_s, y_val_s, y_test_s])

X_seqs_all, y_seqs_all = make_sequences(X_all, y_all, SEQ_LEN)

#   시퀀스 인덱스 경계:
#     X_seqs_all[k].target = y_all[k + SEQ_LEN]
#     train 타겟: k < len(train) - SEQ_LEN
#     val   타겟: k in [len(train)-SEQ_LEN, len(train)-SEQ_LEN+len(val))
#     test  타겟: k >= len(train)-SEQ_LEN+len(val)
n_tr_seqs  = len(X_train) - SEQ_LEN
n_val_seqs = len(X_val)

X_dl_tr    = X_seqs_all[:n_tr_seqs]
y_dl_tr    = y_seqs_all[:n_tr_seqs]
X_dl_val   = X_seqs_all[n_tr_seqs:n_tr_seqs + n_val_seqs]
y_dl_val   = y_seqs_all[n_tr_seqs:n_tr_seqs + n_val_seqs]
X_dl_test  = X_seqs_all[n_tr_seqs + n_val_seqs:]

ptst_final = PatchTST(N_VARS, SEQ_LEN, PATCH_LEN, STRIDE,
                      D_MODEL, N_HEAD, N_LAYERS, DROPOUT)
_train_patchtst(ptst_final, X_dl_tr, y_dl_tr, FINAL_EPOCHS,
                X_dl_val, y_dl_val, PATIENCE)
print('PatchTST 최종 학습 완료')

In [ ]:
# ── PatchTST Val / Test 예측 ──────────────────────────────────────────────
ptst_final.eval()
with torch.no_grad():
    ptst_val_s  = ptst_final(torch.FloatTensor(X_dl_val).to(DEVICE)).cpu().numpy()
    ptst_test_s = ptst_final(torch.FloatTensor(X_dl_test).to(DEVICE)).cpu().numpy()

ptst_val  = ys_all.inverse_transform(ptst_val_s.reshape(-1, 1)).ravel()
ptst_test = ys_all.inverse_transform(ptst_test_s.reshape(-1, 1)).ravel()

# ── GB / LGB Val 예측 ────────────────────────────────────────────────────
gb_val   = gb_final.predict(X_val.values)
lgb_val  = lgb_final.predict(X_val.values)
nd_val   = naive_drift(y_train, len(y_val))

# ── OOF 메타 스택 예측 ───────────────────────────────────────────────────
oof_stack_val  = meta.predict(np.column_stack([nd_val,  gb_val,  lgb_val,  ptst_val]))
oof_stack_test = meta.predict(np.column_stack([nd_test, gb_test, lgb_test, ptst_test]))

# ── 성능 출력 ────────────────────────────────────────────────────────────
print('=== Validation Set 성능 ===')
eval_metrics(y_val, nd_val,                          'Naive Drift')
eval_metrics(y_val, 0.8*nd_val + 0.2*gb_val,        'Hybrid (0.8 Naive + 0.2 GB)')
eval_metrics(y_val, ptst_val,                        'PatchTST')
eval_metrics(y_val, oof_stack_val,                   'OOF Stacking (Meta Ridge)')

print('\n=== Test Set 성능 ===')
results.append(eval_metrics(y_test, ptst_test,       'PatchTST'))
results.append(eval_metrics(y_test, oof_stack_test,  'OOF Stacking (Meta Ridge)'))

## 8. 최종 성능 비교

In [ ]:
df_res = pd.DataFrame(results)
df_res['delta'] = df_res['rmse'] - BASELINE_RMSE
df_res['비고']   = df_res['delta'].apply(
    lambda x: f'▲ {abs(x):.2f} 개선' if x < 0 else f'▼ {x:.2f} 악화'
)
df_res = df_res.set_index('model')

print('=' * 78)
print(f'  최종 성능 비교 (Test Set)     기준선: Hybrid RMSE = {BASELINE_RMSE}')
print('=' * 78)
print(df_res[['rmse', 'mae', 'mape', 'delta', '비고']].to_string())
print('=' * 78)
best = df_res['rmse'].idxmin()
print(f'\n  ★ 최고 성능: {best}  (RMSE={df_res.loc[best, "rmse"]:.2f})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, idx, y_true, preds in [
    (axes[0], y_val.index,  y_val.values,
     [('실제값',             y_val.values,                  'k-o', 2, 5),
      ('Naive Drift',       nd_val,                        'b--', 1, 0),
      ('Hybrid',            0.8*nd_val + 0.2*gb_val,       'g--', 1, 0),
      ('PatchTST',          ptst_val,                      'm-^', 1.5, 4),
      ('OOF Stacking',      oof_stack_val,                 'r-o', 2, 5)]),
    (axes[1], y_test.index, y_test.values,
     [('실제값',             y_test.values,                 'k-o', 2, 5),
      ('Naive Drift',       nd_test,                       'b--', 1, 0),
      ('Hybrid',            0.8*nd_test + 0.2*gb_test,    'g--', 1, 0),
      ('PatchTST',          ptst_test,                     'm-^', 1.5, 4),
      ('OOF Stacking',      oof_stack_test,                'r-o', 2, 5)]),
]:
    for label, vals, ls, lw, ms in preds:
        ax.plot(idx, vals, ls, label=label, linewidth=lw,
                markersize=ms, alpha=(0.7 if '--' in ls else 1.0))
    title = 'Validation Set' if ax is axes[0] else 'Test Set'
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Nickel Price ($/ton)')
    ax.grid(alpha=0.3)

axes[1].set_xlabel('날짜')
plt.suptitle('OOF Stacking + PatchTST vs 베이스라인 (니켈 현물가격, 주간)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. 결론

### OOF Stacking 정확성 보장 포인트

| 항목 | 처리 방법 |
|------|----------|
| 시간 방향 leakage | `TimeSeriesSplit` (과거→미래 방향 고정) |
| 스케일러 leakage | `StandardScaler`를 `fold_train`에만 fit |
| 시퀀스 경계 | `n_tr_seqs = n_tr - SEQ_LEN` 으로 fold_val 타겟과 정확히 1:1 대응 |
| Naive Drift 일관성 | OOF와 인퍼런스 모두 동일한 `y_{t+k} = y_t + k*(y_t - y_{t-1})` 공식 |
| 메타 모델 학습 | OOF 예측(Out-of-sample)만 사용, 인퍼런스는 전체 재학습 모델 적용 |

### PatchTST 특징
- **Patch 토큰화**: 연속된 4주를 하나의 토큰으로 압축 → 지역적 추세 보존
- **Channel-Independence**: 85개 변수 각각을 독립 처리 → 파라미터 공유로 효율화
- **Pre-LN**: 얕은 네트워크(2층)에서도 안정적 학습

### 한계 및 개선 방향
- 주간 12개(Test)의 짧은 평가 구간 → 성능 추정 분산이 큼
- PatchTST OOF 학습 에포크(40)를 고정 사용 → fold별 Early Stopping 검토
- 메타 모델 고도화: Ridge → LightGBM (비선형 가중치 관계 포착)